# Lab 10 -- Decision Trees

Compared to last week, this is a very simple lab <span style="font-size:20pt;">😃</span> You'll have fun programming!

You will implement the **Classification and Regression Tree (CART)** algorithm from scratch.

The lab is broken down into the following pieces:

* Regression Criterion
* Creating Splits
* Buiding a Tree
* Making a prediction


# Decision trees for Regression
## Exercise 1 -- Download and load the dataset

We will be using the usual Boston Housing dataset, which is available to download from ECLASS

* Download the file
* Read it and separate the target variable from the features.
* Make a 80/10/10 train/validation/test split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
np.random.seed(0)

In [ ]:
housing = pd.read_csv("../data/housing.csv")
display(housing.head())

<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\rudab\AppData\Local\Temp\ipykernel_18116\2919588172.py:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  housing = pd.read_table("housing.txt", names=housing_names, sep='\s+')


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


The target variable will be as usual `MEDV`. Use the rest as features.

In [ ]:
X = housing.iloc[:, :-1].values
y = housing["MEDV"].values

In [ ]:
X_train, X_aux, y_train, y_aux = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, test_size=0.5)

## Exercise 2 -- Optimization Criterion

For regression, a simple criterion to optimize is to minimize the sum of squared errors for a given region. This is, for all datapoints in a region with size, we minimize:

$$\sum_{i=1}^N(y_i - \hat{y})^2$$

where $N$ is the number of datapoits in the region and $\hat{y}$ is the mean value of the region for the target variable. 

Implement such a function using the description below.

Please, don't use an existing implementation, refer to the [book](https://www.statlearning.com/s/ISLRSeventhPrinting.pdf), and if you need help, ask questions!

In [ ]:
def regression_criterion(region):
    """
    Implements the sum of squared error criterion in a region
    
    Parameters
    ----------
    region : ndarray
        Array of shape (N,) containing the values of the target values 
        for N datapoints in the training set.
    
    Returns
    -------
    float
        The sum of squared error
        
    Note
    ----
    The error for an empty region should be infinity (use: float("inf"))
    This avoids creating empty regions
    """

    if region.size == 0: return np.inf

    return np.sum((region - region.mean()) ** 2)

In [ ]:
# test your code
rng = np.random.default_rng(0)
print(regression_criterion(rng.random(size=40)))
print(regression_criterion(np.ones(10)))
print(regression_criterion(np.zeros(10)))
print(regression_criterion(np.array([])))

3.6200679838629544
0.0
0.0
inf


## Exercise 3 -- Make a split

In [ ]:
def split_region(region, feature_index, tau):
    """
    Given a region, splits it based on the feature indicated by
    `feature_index`, the region will be split in two, where
    one side will contain all points with the feature with values 
    lower than `tau`, and the other split will contain the 
    remaining datapoints.
    
    Parameters
    ----------
    region : array of size (n_samples, n_features)
        a partition of the dataset (or the full dataset) to be split
    feature_index : int
        the index of the feature (column of the region array) used to make this partition
    tau : float
        The threshold used to make this partition
        
    Return
    ------
    left_partition : array
        indices of the datapoints in `region` where feature < `tau`
    right_partition : array
        indices of the datapoints in `region` where feature >= `tau` 
    """
    
    left_partition = np.where(region[:, feature_index] < tau)[0]
    right_partition = np.where(region[:, feature_index] >= tau)[0]

    return left_partition, right_partition

## Exercise 4 -- Find the best split

The strategy is quite simple (as well as inefficient), but it helps to reinforce the concepts.
We are going to use a greedy, exhaustive algorithm to select splits, selecting the `feature_index` and the `tau` that minimizes the Regression Criterion

In [ ]:
def get_split(X, y):
    """
    Given a dataset (full or partial), splits it on the feature of that minimizes the sum of squared error
    
    Parameters
    ----------
    X : array (n_samples, n_features)
        features 
    y : array (n_samples, )
        labels
    
    Returns
    -------
    decision : dictionary
        keys are:
        * 'feature_index' -> an integer that indicates the feature (column) of `X` on which the data is split
        * 'tau' -> the threshold used to make the split
        * 'left_region' -> array of indices where the `feature_index`th feature of X is lower than `tau`
        * 'right_region' -> indices not in `low_region`
    """
    
    n_features = X.shape[1]
    best_loss = np.inf
    decision = None

    for i in range(n_features):
        u_values = np.unique(X[:, i])
        for tau in u_values:
            l_part, r_part = split_region(X, i, tau)
            loss = regression_criterion(y[l_part]) + regression_criterion(y[r_part])
            if loss < best_loss:
                best_loss = loss
                decision = {'feature_index': i,
                            'tau': tau,
                            'left_region': l_part,
                            'right_region': r_part}
                
    return decision

## Exercise 5 -- Recursive Splitting

The test above is an example on how to find the root node of our decision tree. The algorithm now is a greedy search until we reach a stop criterion. To find the actual root node of our decision tree, you must provide the whole training set, not just a slice of 15 rows as the test above.

The trivial stopping criterion is to recursively grow the tree until each split contains a single point (perfect node purity). If we go that far, it normally means we are overfitting.

You will implement these criteria to stop the growth:

* A node is a leaf if:
    * It has less than `min_samples` datapoints
    * It is at the `max_depth` level from the root (each split creates a new level)
    * The criterion is `0`



In [ ]:
def recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    """
    Recursively grows a decision tree.
    
    Parameters
    ----------
    node : dictionary
        If the node is terminal, it contains only the "value" key, which determines the value to be used as a prediction.
        If the node is not terminal, the dictionary has the structure defined by `get_split`
    min_samples : int
        parameter for stopping criterion if a node has <= min_samples datapoints
    max_depth : int
        parameter for stopping criterion if a node belongs to this depth
    depth : int
        current distance from the root
    X : array (n_samples, n_features)
        features (full dataset)
    y : array (n_samples, )
        labels (full dataset)
    
    Notes
    -----
    To create a terminal node, a dictionary is created with a single "value" key, with a value that
    is the mean of the target variable
    
    'left' and 'right' keys are added to non-terminal nodes, which contain (possibly terminal) nodes 
    from higher levels of the tree:
    'left' corresponds to the 'left_region' key, and 'right' to the 'right_region' key
    """
    
    # the received node is a decision type dictionary as given by get_split(...)
    if X[node['left_region']].shape[0] <= min_samples or current_depth >= max_depth or regression_criterion(y[node['left_region']]) == 0:
        node['left'] = {'value': y[node['left_region']].mean()}
    else: 
        node['left'] = get_split(X[node['left_region']], y[node['left_region']])
        recursive_growth(node['left'], min_samples, max_depth, current_depth+1, X[node['left_region']], y[node['left_region']])

    if X[node['right_region']].shape[0] <= min_samples or current_depth >= max_depth or regression_criterion(y[node['right_region']]) == 0:
        node['right'] = {'value': y[node['right_region']].mean()}
    else: 
        node['right'] = get_split(X[node['right_region']], y[node['right_region']])
        recursive_growth(node['right'], min_samples, max_depth, current_depth+1, X[node['right_region']], y[node['right_region']])

    return

Below we provide code to visualise the generated tree!

In [ ]:
def print_tree(node, depth):
    if 'value' in node.keys():
        print(f'{'.  ' * depth} [{node['value']}]')
        return
    
    print(f'{'.  ' * depth} X_{node['feature_index']} < {node['tau']:.4f}')
    print_tree(node['left'], depth+1)
    print_tree(node['right'], depth+1)

In [ ]:
root = get_split(X_train, y_train)
recursive_growth(root, min_samples=20, max_depth=6, current_depth=0, X=X_train, y=y_train)

In [ ]:
print_tree(root, 0)

 X_12 < 8.1600
.   X_5 < 7.4540
.  .   X_5 < 6.6830
.  .  .   X_0 < 4.8982
.  .  .  .   X_5 < 6.5490
.  .  .  .  .   X_12 < 7.8300
.  .  .  .  .  .   X_6 < 58.7000
.  .  .  .  .  .  .   [23.458064516129035]
.  .  .  .  .  .  .   [25.4625]
.  .  .  .  .  .   [20.459999999999997]
.  .  .  .  .   [27.17142857142857]
.  .  .  .   [50.0]
.  .  .   X_7 < 1.8946
.  .  .  .   [47.1]
.  .  .  .   X_10 < 19.7000
.  .  .  .  .   X_5 < 6.9510
.  .  .  .  .  .   [29.90555555555556]
.  .  .  .  .  .   X_9 < 270.0000
.  .  .  .  .  .  .   [35.37333333333333]
.  .  .  .  .  .  .   [32.36666666666666]
.  .  .  .  .   [24.8]
.  .   X_10 < 18.6000
.  .  .   X_10 < 14.9000
.  .  .  .   [48.169230769230765]
.  .  .  .   [43.88888888888889]
.  .  .   [28.55]
.   X_12 < 15.0200
.  .   X_5 < 6.6160
.  .  .   X_2 < 3.2400
.  .  .  .   [26.860000000000003]
.  .  .  .   X_11 < 227.6100
.  .  .  .  .   [13.5]
.  .  .  .  .   X_6 < 76.5000
.  .  .  .  .  .   X_5 < 6.0150
.  .  .  .  .  .  .   [20.65714285714286]
.

## Exercise 6 -- Make a Prediction
Use the a node to predict the class of a compatible dataset

In [ ]:
def predict_sample(node, sample):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    sample : array of size (n_features,)
        a sample datapoint
    """
    
    if 'value' in node.keys():
        return node['value']
    
    if sample[node['feature_index']] < node['tau']:
        return predict_sample(node['left'], sample)
    else:
        return predict_sample(node['right'], sample)
        
def predict(node, X):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    X : array of size (n_samples, n_features)
        n_samples predictions will be made
    """
    
    return np.array([predict_sample(node, s) for s in X])

Now use the functions defined above to calculate the RMSE of the validation set. 
* Try first with `min_samples=20` and `max_depth=6` (for this values you should get a validation RMSE of ~8.8)

Then, experiment with different values for the stopping criteria.

In [ ]:
# calculate root mean squared error with numpy
def root_mean_squared_error(y_true, y_pred):
    """
    Calculates the root mean squared error between two arrays
    
    Parameters
    ----------
    y_true : array of size (n_samples,)
        true labels
    y_pred : array of size (n_samples,)
        predicted labels
    """
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [ ]:
root = get_split(X_train, y_train)
min_samples = 20
max_depth = 6
recursive_growth(root, min_samples, max_depth, 1, X_train, y_train)
train_mse = root_mean_squared_error(y_train, predict(root, X_train))
test_mse = root_mean_squared_error(y_test, predict(root, X_test))

print(f'Train MSE : {train_mse}')
print(f'Test MSE : {test_mse}')

Train MSE : 2.531441402661717
Test MSE : 6.96288046111963


## Just to see how much we can improve this naive decision tree!

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
reg = LinearRegression().fit(X_train, y_train)

In [ ]:
reg_train_mse = root_mean_squared_error(y_train, reg.predict(X_train))
reg_test_mse = root_mean_squared_error(y_test, reg.predict(X_test))
print(f'Regression Train MSE : {reg_train_mse}')
print(f'Regression Test MSE : {reg_test_mse}')

Regression Train MSE : 4.396188144698282
Regression Test MSE : 6.657184354566778


# Decision trees for Classification
You will implement decision trees for classification using the Gini index as the splitting criterion. You’ll build the tree recursively, selecting splits that minimize Gini impurity and classifying samples based on majority class in each leaf. A good dataset to start with is the Iris dataset, which is small, well-labeled, and available directly via `sklearn.datasets.load_iris().`

In [ ]:
from sklearn.datasets import load_iris

In [ ]:
# Load the Iris dataset
data = load_iris()
X = data.data
y = data.target

We will focus only on binary classification today!

In [ ]:
X = X[y != 2]
y = y[y != 2]

In [ ]:
# split your data into training, validation and test sets!

X_train, X_aux, y_train, y_aux = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, test_size=0.5)

### Feel free to use the same code as for regression, but change the criterion and the prediction function. You can also implement a new one if you want to!

In [ ]:
def classification_criterion(region):
    """
    Implements the sum of squared error criterion in a region
    
    Parameters
    ----------
    region : ndarray
        Array of shape (N,) containing the values of the target values 
        for N datapoints in the training set.
    
    Returns
    -------
    float
        The gini coeficient
        
    Note
    ----
    The error for an empty region should be infinity (use: float("inf"))
    This avoids creating empty regions
    """

    if region.size == 0: return np.inf

    counts = np.unique(y, return_counts=True)[1]
    pmk = counts / counts.sum()
    gini = pmk * (1 - pmk)
    return gini.sum()

In [ ]:
def classification_get_split(X, y):
    """
    Given a dataset (full or partial), splits it on the feature of that maximize the gain
    
    Parameters
    ----------
    X : array (n_samples, n_features)
        features 
    y : array (n_samples, )
        labels
    
    Returns
    -------
    decision : dictionary
        keys are:
        * 'feature_index' -> an integer that indicates the feature (column) of `X` on which the data is split
        * 'tau' -> the threshold used to make the split
        * 'left_region' -> array of indices where the `feature_index`th feature of X is lower than `tau`
        * 'right_region' -> indices not in `low_region`
    """
    
    n_features = X.shape[1]
    best_gain = 0
    parent_gini = classification_criterion(y)
    S = X.shape[0]
    decision = None

    for i in range(n_features):
        u_values = np.unique(X[:, i])
        for tau in u_values:
            l_part, r_part = split_region(X, i, tau)
            l_gini = (l_part.size / S) * classification_criterion(y[l_part])
            r_gini = (r_part.size / S) * classification_criterion(y[r_part])
            gain = parent_gini - l_gini - r_gini
            if gain > best_gain:
                best_gain = gain
                decision = {'feature_index': i,
                            'tau': tau,
                            'left_region': l_part,
                            'right_region': r_part}
                
    return decision

In [ ]:
def classification_recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    """
    Recursively grows a decision tree.
    
    Parameters
    ----------
    node : dictionary
        If the node is terminal, it contains only the "value" key, which determines the value to be used as a prediction.
        If the node is not terminal, the dictionary has the structure defined by `get_split`
    min_samples : int
        parameter for stopping criterion if a node has <= min_samples datapoints
    max_depth : int
        parameter for stopping criterion if a node belongs to this depth
    depth : int
        current distance from the root
    X : array (n_samples, n_features)
        features (full dataset)
    y : array (n_samples, )
        labels (full dataset)
    
    Notes
    -----
    To create a terminal node, a dictionary is created with a single "value" key, with a value that
    is the mean of the target variable
    
    'left' and 'right' keys are added to non-terminal nodes, which contain (possibly terminal) nodes 
    from higher levels of the tree:
    'left' corresponds to the 'left_region' key, and 'right' to the 'right_region' key
    """
    
    # the received node is a decision type dictionary as given by get_split(...)
    if X[node['left_region']].shape[0] <= min_samples or current_depth >= max_depth or classification_criterion(y[node['left_region']]) == 0:
        node['left'] = {'value': y[node['left_region']].mean()}
    else: 
        node['left'] = get_split(X[node['left_region']], y[node['left_region']])
        classification_recursive_growth(node['left'], min_samples, max_depth, current_depth+1, X[node['left_region']], y[node['left_region']])

    if X[node['right_region']].shape[0] <= min_samples or current_depth >= max_depth or classification_criterion(y[node['right_region']]) == 0:
        node['right'] = {'value': y[node['right_region']].mean()}
    else: 
        node['right'] = get_split(X[node['right_region']], y[node['right_region']])
        classification_recursive_growth(node['right'], min_samples, max_depth, current_depth+1, X[node['right_region']], y[node['right_region']])

    return

In [ ]:
# you will test 3 values for min_samples_split: 2, 4, 6
# Remember that this sets the minimum number of samples required in a node to be eligible for splitting. 
# These values are good for small datasets like Iris, but you can try other values for larger datasets to not make the tree too deep.

# your code goes here
def cross_entropy(y_true, y_pred):
    c_y = np.clip(y_pred, 1e-8, 1 - 1e-8)
    return - np.sum(y_true * np.log(c_y) + (1 - y_true) * np.log(1 - c_y))

models = []
for ms in [2, 4, 6]:
    root = classification_get_split(X_train, y_train)
    classification_recursive_growth(root, min_samples=ms, max_depth=5, current_depth=0, X=X_train, y=y_train)

    y_pred_val = predict(root, X_val)
    loss = cross_entropy(y_val, y_pred_val)
    acc = (y_val == y_pred_val).astype(int).mean()

    models.append({'min_samples':ms, 
                   'model': root,
                   'loss': loss, 
                   'accuracy': acc})

print('total training samples: ', y_train.shape[0])
mdf = pd.DataFrame(models)
display(mdf)

total training samples:  80


,min_samples,model,loss,accuracy
0,2,"{'feature_index': 0, 'tau': 6.0, 'left_region'...",1.000000e-07,1.0
1,4,"{'feature_index': 0, 'tau': 6.0, 'left_region'...",1.000000e-07,1.0
2,6,"{'feature_index': 0, 'tau': 6.0, 'left_region'...",1.000000e-07,1.0


Use accuracy to find the best split. Don't import it from sklearn, calculate it yourself, it's a one-liner ;)

In [ ]:
## Lastly, evaluate your model on the test set and print the accuracy score

# your code goes here
